<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E08/mva_2026_E08_01_gmm_EM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E08_01_gmm-EM.ipynb

- 学籍番号 [ ]
- 氏名 [ ]

In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

from sklearn.mixture import GaussianMixture # GMMのEMアルゴリズム

%pip install japanize_matplotlib
import japanize_matplotlib # 日本語フォント

## 混合ガウス分布にしたがうらしい未知のデータ

In [ ]:
df=pd.read_csv("https://www.data.math.ryukoku.ac.jp/course/mva_2026/data/gmm2d_unlabeled.csv")
df

In [ ]:
# import matplotlib.pyplot as plt

# 散布図
plt.figure(figsize=(8, 6))
plt.scatter(df['x1'], df['x2'], alpha=0.6, edgecolors='k', c='gray')
plt.title('散布図')
plt.xlabel('x1')
plt.ylabel('x2')
plt.grid(True)
plt.show()

### モデル

ガウス混合モデルで，EMアルゴリズムによるクラスタリングを使用．

In [ ]:
# from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=3) # これでも初期値を適当に決めて実行される(n_componentsのデフォルト値は無意味な1)

初期値の設定

In [ ]:
K=3

# 3つのクラスタの初期中心（平均値）mu_k
initial_means = np.array([
    [-2.0, -2.0],  # クラスタ0の初期中心
    [ 2.0,  2.0],  # クラスタ1の初期中心
    [ 0.0,  4.0]   # クラスタ2の初期中心
])

# 3つのクラスタの初期共分散行列の逆数 S_k^{-1}
initial_precisions=np.array([
   [[1,0],[0,1]],
    [[1,0],[0,1]],
    [[1,0],[0,1]]
  ])

# 3つのクラスタの初期重み pi_k
initial_weights=np.array([
   0.1,0.2,0.7
  ])

In [ ]:
# 1. まず初期値を反映したGMMのインスタンスを作成します（n_components=K もここで指定します）
gmm = GaussianMixture(
    n_components=K,
    means_init=initial_means,
    precisions_init=initial_precisions,
    weights_init=initial_weights,
    max_iter=100
)

In [ ]:
gmm.fit(df)

In [ ]:

# 各データ点のクラス（予測ラベル）と、各クラスへの責任度（事後確率）を取得
labels = gmm.predict(df)
probs = gmm.predict_proba(df)
print(labels[:10])
print(probs[:10])

In [ ]:
# 学習後の平均値を出力して確認
print("指定した重み:\n", initial_weights)
print("終了時の重み:\n", gmm.weights_)

# 学習後の平均値を出力して確認
print("指定した初期値:\n", initial_means)
print("学習後の平均値:\n", gmm.means_)

### 可視化

In [ ]:
# x1,x2,label,r1,r2,r3をまとめた，可視化専用データフレームを作る
df_result = df.copy()
df_result['label'] = labels
for i in range(gmm.n_components):
    df_result[f'r_{i}'] = probs[:, i]

# 結果の表示
display(df_result.head())

In [ ]:
# クラスタリング結果の可視化．散布図．
plt.figure(figsize=(8, 6))
scatter = plt.scatter(df['x1'], df['x2'], c=labels, cmap='viridis', alpha=0.7, edgecolors='b')
plt.title('GMM EMアルゴリズムによるクラスタリング結果')
plt.xlabel('x1')
plt.ylabel('x2')
plt.colorbar(scatter, label='Cluster Label')
plt.grid(True)
plt.show()

### 評価

In [ ]:
# データの平均対数尤度を計算
log_likelihood = gmm.score(df)
print(f"平均対数尤度（大きい方がよい）: {log_likelihood:.4f}")

# AIC / BIC による評価指標もあわせて確認
print(f"AIC（小さい方がよい）: {gmm.aic(df):.4f}")
print(f"BIC（小さい方がよい）: {gmm.bic(df):.4f}")

## 課題
K=2,4にして同様の分析をしてみよう

## 課題
初期値を変えて同様の分析をしてみよう